In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = "Malgun Gothic"

BASE_DIR = Path(".")
RESULT_DIR = Path("final_mismatch_results")
RESULT_DIR.mkdir(exist_ok=True)

for sub in [
    "01_youth_outflow_analysis",
    "02_policy_mismatch_score",
    "03_mismatch_textmining",
]:
    (RESULT_DIR / sub).mkdir(parents=True, exist_ok=True)


def read_csv_auto(path):
    for enc in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    raise ValueError(f"CSV 파일을 읽을 수 없습니다: {path}")


# 1. 파일 불러오기
migration = read_csv_auto("youth_migration.csv")
supply = read_csv_auto("final_results/01_region_policy_supply_score/region_policy_supply_score_rank.csv")
gap = read_csv_auto("final_results/02_region_policy_gap_score/region_policy_gap_rank.csv")

print("migration 컬럼:", migration.columns.tolist())
print("supply 컬럼:", supply.columns.tolist())
print("gap 컬럼:", gap.columns.tolist())

display(migration.head())

migration 컬럼: ['청년유출순위', '지역', '기준연도', '기준기간', '연앙인구기준연도', '청년연령기준', '청년순이동자수', '청년연앙인구', '청년순이동률', '청년유출위험도', '유출여부', '연앙인구산출방식']
supply 컬럼: ['지역', '정책수', '일자리문제관련정책수', '평균키워드대응점수', '평균문제대응개수', '신청가능비율', '신청URL보유비율', '신청방법보유비율', '일자리문제관련정책비율', '정보접근성비율', '분류개수', '분류다양성점수', '정책수점수', '관련정책비율점수', '키워드대응도점수', '신청가능성점수', '정보접근성점수', '정책공급종합점수', '정책대응부족도', '정책대응부족분야', '정책공급수준', '해석']
gap 컬럼: ['지역', '정책수', '일자리문제관련정책수', '평균키워드대응점수', '평균문제대응개수', '신청가능비율', '신청URL보유비율', '신청방법보유비율', '일자리문제관련정책비율', '정보접근성비율', '분류개수', '분류다양성점수', '정책수점수', '관련정책비율점수', '키워드대응도점수', '신청가능성점수', '정보접근성점수', '정책공급종합점수', '정책대응부족도', '정책대응부족분야', '정책공급수준', '해석']


,청년유출순위,지역,기준연도,기준기간,연앙인구기준연도,청년연령기준,청년순이동자수,청년연앙인구,청년순이동률,청년유출위험도,유출여부,연앙인구산출방식
0,1,경북,2025.11~2026.04,2025.11~2026.04,2025,19~34세,-6054,377923.1,-1.6019,100.00,유출,15~19세의 1/5 + 20~24세 + 25~29세 + 30~34세
1,2,전북,2025.11~2026.04,2025.11~2026.04,2025,19~34세,-3225,283040.9,-1.1394,81.78,유출,15~19세의 1/5 + 20~24세 + 25~29세 + 30~34세
2,3,제주,2025.11~2026.04,2025.11~2026.04,2025,19~34세,-1273,114625.9,-1.1106,80.64,유출,15~19세의 1/5 + 20~24세 + 25~29세 + 30~34세
3,4,경남,2025.11~2026.04,2025.11~2026.04,2025,19~34세,-5519,500754.7,-1.1021,80.31,유출,15~19세의 1/5 + 20~24세 + 25~29세 + 30~34세
4,5,전남,2025.11~2026.04,2025.11~2026.04,2025,19~34세,-2921,267669.0,-1.0913,79.88,유출,15~19세의 1/5 + 20~24세 + 25~29세 + 30~34세


In [4]:
# 필요한 컬럼 자동 탐색
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"컬럼을 찾지 못했습니다. 후보={candidates}, 실제={df.columns.tolist()}")


region_col_m = find_col(migration, ["지역", "region"])
migration_rate_col = find_col(migration, ["청년순이동률", "순이동률"])
migration_count_col = find_col(migration, ["청년순이동자수", "순이동자수"])

region_col_s = find_col(supply, ["지역", "region"])
supply_score_col = find_col(supply, ["정책공급종합점수", "정책공급점수", "종합점수"])

region_col_g = find_col(gap, ["지역", "region"])

# 정책대응부족도 컬럼 찾기
gap_candidates = ["정책대응부족도", "정책부족도", "대응부족도", "정책대응부족도_계산"]
gap_score_col = None
for c in gap_candidates:
    if c in gap.columns:
        gap_score_col = c
        break

# 없으면 정책공급종합점수에서 계산
if gap_score_col is None:
    if supply_score_col in gap.columns:
        gap["정책대응부족도"] = 100 - pd.to_numeric(gap[supply_score_col], errors="coerce")
        gap_score_col = "정책대응부족도"
    else:
        raise KeyError("정책대응부족도 컬럼을 찾지 못했습니다.")

# 필요한 컬럼만 추출
migration_small = migration[[region_col_m, migration_count_col, migration_rate_col]].copy()
migration_small.columns = ["지역", "청년순이동자수", "청년순이동률"]

supply_small = supply[[region_col_s, supply_score_col]].copy()
supply_small.columns = ["지역", "정책공급종합점수"]

gap_small = gap[[region_col_g, gap_score_col]].copy()
gap_small.columns = ["지역", "정책대응부족도"]

# 병합
merged = migration_small.merge(supply_small, on="지역", how="left")
merged = merged.merge(gap_small, on="지역", how="left")

# 숫자 변환
for col in ["청년순이동자수", "청년순이동률", "정책공급종합점수", "정책대응부족도"]:
    merged[col] = pd.to_numeric(merged[col], errors="coerce")

# 청년유출강도: 음수 순이동률만 위험으로 반영
merged["청년유출강도"] = (-merged["청년순이동률"]).clip(lower=0)

# 0~100 정규화
min_val = merged["청년유출강도"].min()
max_val = merged["청년유출강도"].max()

if max_val == min_val:
    merged["청년유출위험도"] = 0
else:
    merged["청년유출위험도"] = (
        (merged["청년유출강도"] - min_val) / (max_val - min_val) * 100
    )

# 미스매치 점수
merged["정책미스매치점수"] = (
    merged["청년유출위험도"] * merged["정책대응부족도"] / 100
)

# 순위
merged["청년유출위험순위"] = merged["청년유출위험도"].rank(ascending=False, method="min").astype(int)
merged["정책미스매치순위"] = merged["정책미스매치점수"].rank(ascending=False, method="min").astype(int)

# 정렬
merged = merged.sort_values("정책미스매치점수", ascending=False)

# 저장
out_path = RESULT_DIR / "02_policy_mismatch_score" / "region_policy_mismatch_score.csv"
merged.to_csv(out_path, index=False, encoding="utf-8-sig")

display(merged)

print("저장 완료:", out_path)

,지역,청년순이동자수,청년순이동률,정책공급종합점수,정책대응부족도,청년유출강도,청년유출위험도,정책미스매치점수,청년유출위험순위,정책미스매치순위
1,전북,-3225,-1.1394,46.40,53.60,1.1394,71.128035,38.124627,2,1
4,전남,-2921,-1.0913,49.19,50.81,1.0913,68.125351,34.614491,5,2
3,경남,-5519,-1.1021,63.52,36.48,1.1021,68.799551,25.098076,4,3
0,경북,-6054,-1.6019,75.90,24.10,1.6019,100.000000,24.100000,1,4
2,제주,-1273,-1.1106,75.21,24.79,1.1106,69.330170,17.186949,3,5
5,강원,-817,-0.3302,52.31,47.69,0.3302,20.613022,9.830350,6,6
6,충남,-232,-0.0639,46.14,53.86,0.0639,3.989013,2.148482,7,7
7,경기,8697,0.3204,62.38,37.62,0.0000,0.000000,0.000000,8,8
8,서울,15631,0.7265,60.67,39.33,0.0000,0.000000,0.000000,8,8
9,충북,2672,0.9364,62.44,37.56,0.0000,0.000000,0.000000,8,8


저장 완료: final_mismatch_results\02_policy_mismatch_score\region_policy_mismatch_score.csv


In [5]:
risk_median = merged["청년유출위험도"].median()
gap_median = merged["정책대응부족도"].median()

def classify_quadrant(row):
    high_risk = row["청년유출위험도"] >= risk_median
    high_gap = row["정책대응부족도"] >= gap_median
    
    if high_risk and high_gap:
        return "고유출-고부족"
    elif high_risk and not high_gap:
        return "고유출-저부족"
    elif not high_risk and high_gap:
        return "저유출-고부족"
    else:
        return "저유출-저부족"

merged["미스매치유형"] = merged.apply(classify_quadrant, axis=1)

quadrant_path = RESULT_DIR / "02_policy_mismatch_score" / "region_policy_mismatch_quadrant.csv"
merged.to_csv(quadrant_path, index=False, encoding="utf-8-sig")

display(merged[["지역", "청년유출위험도", "정책대응부족도", "정책미스매치점수", "미스매치유형"]])

,지역,청년유출위험도,정책대응부족도,정책미스매치점수,미스매치유형
1,전북,71.128035,53.60,38.124627,고유출-고부족
4,전남,68.125351,50.81,34.614491,고유출-고부족
3,경남,68.799551,36.48,25.098076,고유출-저부족
0,경북,100.000000,24.10,24.100000,고유출-저부족
2,제주,69.330170,24.79,17.186949,고유출-저부족
5,강원,20.613022,47.69,9.830350,저유출-고부족
6,충남,3.989013,53.86,2.148482,저유출-고부족
7,경기,0.000000,37.62,0.000000,저유출-저부족
8,서울,0.000000,39.33,0.000000,저유출-고부족
9,충북,0.000000,37.56,0.000000,저유출-저부족


In [7]:
# 1. 청년순이동률 순위 그래프
plot_df = merged.sort_values("청년순이동률", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["지역"], plot_df["청년순이동률"])
plt.title("지역별 청년순이동률")
plt.xlabel("청년순이동률(%)")
plt.tight_layout()
plt.savefig(RESULT_DIR / "01_youth_outflow_analysis" / "youth_migration_rate_rank.png", dpi=200)
plt.close()

# 2. 정책미스매치점수 그래프
plot_df = merged.sort_values("정책미스매치점수", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["지역"], plot_df["정책미스매치점수"])
plt.title("지역별 청년 유출-정책 미스매치 점수")
plt.xlabel("정책미스매치점수")
plt.tight_layout()
plt.savefig(RESULT_DIR / "02_policy_mismatch_score" / "policy_mismatch_score_rank.png", dpi=200)
plt.close()

# 3. 4분면 산점도
plt.figure(figsize=(8, 6))
plt.scatter(merged["청년유출위험도"], merged["정책대응부족도"])

for _, row in merged.iterrows():
    plt.text(row["청년유출위험도"], row["정책대응부족도"], row["지역"], fontsize=9)

plt.axvline(risk_median, linestyle="--")
plt.axhline(gap_median, linestyle="--")
plt.title("청년 유출위험도-정책대응부족도 4분면 분석")
plt.xlabel("청년유출위험도")
plt.ylabel("정책대응부족도")
plt.tight_layout()
plt.savefig(RESULT_DIR / "02_policy_mismatch_score" / "mismatch_quadrant_scatter.png", dpi=200)
plt.close()

print("그래프 저장 완료")

그래프 저장 완료


In [9]:
from pathlib import Path
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = "Malgun Gothic"

BASE_DIR = Path(".")
RESULT_DIR = Path("final_mismatch_results")
TEXTMINING_DIR = RESULT_DIR / "03_mismatch_textmining"
TEXTMINING_DIR.mkdir(parents=True, exist_ok=True)

print("텍스트마이닝 결과 저장 폴더:", TEXTMINING_DIR.resolve())


def read_csv_auto(path):
    for enc in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    raise ValueError(f"CSV 파일을 읽을 수 없습니다: {path}")


def find_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"필요 컬럼을 찾지 못했습니다. 후보={candidates}, 실제컬럼={list(df.columns)}")
    return None


def normalize_region_name(x):
    if pd.isna(x):
        return ""
    
    x = str(x).strip()
    
    mapping = {
        "서울특별시": "서울",
        "부산광역시": "부산",
        "대구광역시": "대구",
        "인천광역시": "인천",
        "광주광역시": "광주",
        "대전광역시": "대전",
        "울산광역시": "울산",
        "세종특별자치시": "세종",
        "경기도": "경기",
        "강원특별자치도": "강원",
        "강원도": "강원",
        "충청북도": "충북",
        "충청남도": "충남",
        "전북특별자치도": "전북",
        "전라북도": "전북",
        "전라남도": "전남",
        "경상북도": "경북",
        "경상남도": "경남",
        "제주특별자치도": "제주",
    }
    
    for full, short in mapping.items():
        x = x.replace(full, short)
    
    return x


def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^가-힣A-Za-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# =========================================================
# 1. 파일 불러오기
# =========================================================

policy_path_candidates = [
    Path("youth_policies_categorized.csv"),
    Path("outputs/policy_preprocessed.csv"),
    Path("outputs/policy_scored.csv"),
]

policy_path = None
for p in policy_path_candidates:
    if p.exists():
        policy_path = p
        break

if policy_path is None:
    raise FileNotFoundError("youth_policies_categorized.csv 또는 outputs/policy_preprocessed.csv 파일을 찾지 못했습니다.")

mismatch_path = RESULT_DIR / "02_policy_mismatch_score" / "region_policy_mismatch_quadrant.csv"

if not mismatch_path.exists():
    raise FileNotFoundError("region_policy_mismatch_quadrant.csv 파일이 없습니다. 먼저 미스매치 점수 계산과 4분면 분석을 실행해야 합니다.")

policies = read_csv_auto(policy_path)
mismatch = read_csv_auto(mismatch_path)

print("정책 데이터:", policy_path)
print("정책 데이터 컬럼:", policies.columns.tolist())
print("미스매치 데이터 컬럼:", mismatch.columns.tolist())

텍스트마이닝 결과 저장 폴더: C:\Users\yong\Desktop\TM\tm2\textmining_solo_policy_project\textmining_solo_policy_project\final_mismatch_results\03_mismatch_textmining
정책 데이터: youth_policies_categorized.csv
정책 데이터 컬럼: ['지역', '조회_zipCd', '정책ID', '정책명', '정책키워드', '정책설명', '정책지원내용', '정책대분류', '정책중분류', '자동분류', '대표분류', '신청기간', '사업기간', '지원대상', '나이조건', '소득조건', '신청방법', '제출서류', '주관기관', '운영기관', '신청URL', '참고URL1', '참고URL2', '최초등록일시', '최종수정일시', '수집페이지', '수집출처']
미스매치 데이터 컬럼: ['지역', '청년순이동자수', '청년순이동률', '정책공급종합점수', '정책대응부족도', '청년유출강도', '청년유출위험도', '정책미스매치점수', '청년유출위험순위', '정책미스매치순위', '미스매치유형']


In [10]:
# =========================================================
# 2. 미스매치 분석 대상 지역 선정
# =========================================================

m_region_col = find_col(mismatch, ["지역", "region"])
m_score_col = find_col(mismatch, ["정책미스매치점수", "mismatch_score", "미스매치점수"], required=False)
m_type_col = find_col(mismatch, ["미스매치유형", "유형"], required=False)

mismatch[m_region_col] = mismatch[m_region_col].apply(normalize_region_name)

if m_type_col is not None:
    target_regions = (
        mismatch[mismatch[m_type_col] == "고유출-고부족"][m_region_col]
        .dropna()
        .unique()
        .tolist()
    )
else:
    target_regions = []

if len(target_regions) == 0:
    if m_score_col is None:
        raise KeyError("고유출-고부족 지역도 없고, 정책미스매치점수 컬럼도 없습니다.")
    
    target_regions = (
        mismatch.sort_values(m_score_col, ascending=False)
        .head(3)[m_region_col]
        .dropna()
        .unique()
        .tolist()
    )

print("텍스트마이닝 분석 대상 지역:", target_regions)

target_region_df = mismatch[mismatch[m_region_col].isin(target_regions)].copy()
target_region_df.to_csv(
    TEXTMINING_DIR / "target_mismatch_regions.csv",
    index=False,
    encoding="utf-8-sig"
)

display(target_region_df)

텍스트마이닝 분석 대상 지역: ['전북', '전남']


,지역,청년순이동자수,청년순이동률,정책공급종합점수,정책대응부족도,청년유출강도,청년유출위험도,정책미스매치점수,청년유출위험순위,정책미스매치순위,미스매치유형
0,전북,-3225,-1.1394,46.40,53.60,1.1394,71.128035,38.124627,2,1,고유출-고부족
1,전남,-2921,-1.0913,49.19,50.81,1.0913,68.125351,34.614491,5,2,고유출-고부족


In [11]:
# =========================================================
# 3. 미스매치 지역 정책 데이터 추출
# =========================================================

policy_region_col = find_col(
    policies,
    ["지역", "region", "시도", "정책지역", "지역명"]
)

policy_name_col = find_col(
    policies,
    ["정책명", "사업명", "제목", "policy_name", "name"],
    required=False
)

policy_desc_col = find_col(
    policies,
    ["정책설명", "정책내용", "사업내용", "설명", "내용", "policy_desc", "description"],
    required=False
)

policy_keyword_col = find_col(
    policies,
    ["정책키워드", "키워드", "검색키워드", "keyword", "keywords"],
    required=False
)

policy_category_col = find_col(
    policies,
    ["대표분류", "정책분류", "분류", "정책분야", "category"],
    required=False
)

policies["지역_정규화"] = policies[policy_region_col].apply(normalize_region_name)

text_cols = []
for c in [policy_name_col, policy_desc_col, policy_keyword_col, policy_category_col]:
    if c is not None:
        text_cols.append(c)

if len(text_cols) == 0:
    raise KeyError("정책 텍스트 분석에 사용할 정책명/설명/키워드 컬럼을 찾지 못했습니다.")

policies["분석텍스트"] = (
    policies[text_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .apply(clean_text)
)

target_policies = policies[policies["지역_정규화"].isin(target_regions)].copy()

if len(target_policies) == 0:
    raise ValueError("미스매치 대상 지역에 해당하는 정책 데이터가 없습니다. 지역명 매칭을 확인해야 합니다.")

target_policies.to_csv(
    TEXTMINING_DIR / "high_mismatch_policy_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("미스매치 지역 정책 수:", len(target_policies))
display(target_policies.head())

미스매치 지역 정책 수: 1069


,지역,조회_zipCd,정책ID,정책명,정책키워드,정책설명,정책지원내용,정책대분류,정책중분류,자동분류,...,운영기관,신청URL,참고URL1,참고URL2,최초등록일시,최종수정일시,수집페이지,수집출처,지역_정규화,분석텍스트
2670,전북,"52000,45000",20260605005400113228,청년미래적금,보조금,"청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월...","은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 6%, 우대형 12%의 정...",금융･복지･문화,취약계층 및 금융지원,복지,...,한국고용정보원,NaN,https://www.kinfa.or.kr/financialProduct/youth...,https://blog.naver.com/blogfsc/224302863400,2026-06-05 18:06:49,2026-06-10 14:05:31,1,온통청년_OPEN_API_getPlcy,전북,청년미래적금 청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로 3년 만기 ...
2671,전북,"52000,45000",20260528005400113227,(농식품부) 농식품 바우처,바우처,『농업·농촌 및 식품산업 기본법』 제 23조의 2(취약계층 등에 대한 식품지원)에 ...,"- 지원방식: 전자바우처(카드방식) - 지원품목: 국산 채소류, 과일류, 육류, 신...",금융･복지･문화,건강,복지,...,한국고용정보원,https://www.foodvoucher.go.kr/security/joinAgree,https://www.foodvoucher.go.kr/view/fm/vucintro...,NaN,2026-05-28 10:00:50,2026-05-28 10:01:12,1,온통청년_OPEN_API_getPlcy,전북,농식품부 농식품 바우처 농업 농촌 및 식품산업 기본법 제 23조의 2 취약계층 등에...
2672,전북,"52000,45000",20260528005400113226,(문체부) 청년예술인 예술활동 적립계좌,보조금,청년예술인에게 중장기 자산형성의 기회를 마련하여 안정적인 예술활동을 지원하는 사업,매월 일정 금액을 24개월간 적금 저축 시 가입자가 저축한 금액만큼 정부지원금을 지...,금융･복지･문화,예술인지원,복지,...,한국고용정보원,https://www.artloan.kr/notice/savingsAccountPr...,https://www.artloan.kr/notice/savingsAccount.do,NaN,2026-05-28 09:34:16,2026-05-28 09:34:49,1,온통청년_OPEN_API_getPlcy,전북,문체부 청년예술인 예술활동 적립계좌 청년예술인에게 중장기 자산형성의 기회를 마련하여...
2673,전북,"52000,45000",20260527005400113224,청년 국가기술자격 응시료 지원 사업,보조금,"구직활동을 하거나 경력을 개발하는 청년들의 경제적 부담을 완화하고, 국가기술자격 취...",34세 이하 청년*이(소득 및 취업 여부 무관) 한국산업인력공단이 시행하는 국가기술...,일자리,취업,일자리,...,한국산업인력공단,https://www.q-net.or.kr/man001.do?gSite=Q&gInt...,https://www.q-net.or.kr/man004.do?id=man00402&...,NaN,2026-05-27 11:16:43,2026-05-27 11:17:06,1,온통청년_OPEN_API_getPlcy,전북,청년 국가기술자격 응시료 지원 사업 구직활동을 하거나 경력을 개발하는 청년들의 경제...
2674,전북,"52000,45000",20260527005400113223,전세보증금반환보증 보증료 지원,주거지원,"전세사기, 역전세 등 임차인이 전세보증금을 돌려받지 못하는 전세 피해를 예방하고, ...","○ 지원대상 - 신청일 기준 유효한 전세보증금반환보증(HUG, HF, SGI)에 가...",주거,전월세 및 주거급여 지원,"주거지원, 복지",...,국토교통부,https://www.gov.kr/portal/rcvfvrSvc/dtlEx/1613...,https://www.gov.kr/portal/rcvfvrSvc/dtlEx/1613...,NaN,2026-05-27 11:06:12,2026-05-27 11:06:36,1,온통청년_OPEN_API_getPlcy,전북,전세보증금반환보증 보증료 지원 전세사기 역전세 등 임차인이 전세보증금을 돌려받지 못...


In [12]:
# =========================================================
# 4. 키워드 빈도 분석
# =========================================================

stopwords = set([
    "청년", "지원", "사업", "정책", "대상", "신청", "제공", "운영", "관련",
    "지역", "프로그램", "서비스", "모집", "참여", "대상자", "가능", "추진",
    "서울", "경기", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주",
    "특별시", "광역시", "특별자치도", "전라남도", "전라북도", "경상북도", "경상남도",
    "충청북도", "충청남도", "경기도"
])

def token_pattern_text(text):
    tokens = re.findall(r"[가-힣A-Za-z0-9]{2,}", str(text))
    tokens = [t for t in tokens if t not in stopwords]
    return " ".join(tokens)

target_policies["분석텍스트_정제"] = target_policies["분석텍스트"].apply(token_pattern_text)

count_vectorizer = CountVectorizer(
    max_features=100,
    token_pattern=r"(?u)\b[가-힣A-Za-z0-9]{2,}\b"
)

count_matrix = count_vectorizer.fit_transform(target_policies["분석텍스트_정제"])

try:
    terms = count_vectorizer.get_feature_names_out()
except:
    terms = count_vectorizer.get_feature_names()

term_counts = np.asarray(count_matrix.sum(axis=0)).ravel()

keyword_freq = pd.DataFrame({
    "키워드": terms,
    "빈도": term_counts
}).sort_values("빈도", ascending=False)

keyword_freq.to_csv(
    TEXTMINING_DIR / "high_mismatch_keyword_frequency.csv",
    index=False,
    encoding="utf-8-sig"
)

display(keyword_freq.head(30))

# 그래프 저장
top_keywords = keyword_freq.head(20).sort_values("빈도", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(top_keywords["키워드"], top_keywords["빈도"])
plt.title("미스매치 지역 정책 텍스트 상위 키워드")
plt.xlabel("빈도")
plt.tight_layout()
plt.savefig(TEXTMINING_DIR / "high_mismatch_keyword_frequency.png", dpi=200)
plt.close()

print("키워드 빈도 분석 완료")

,키워드,빈도
55,일자리,650
9,교육지원,358
76,직무교육,256
33,보조금,247
70,중소기업,144
46,위한,140
27,맞춤형상담서비스,137
34,복지,127
68,주거지원,121
92,통해,117


키워드 빈도 분석 완료


In [13]:
# =========================================================
# 5. 미스매치 지역별 TF-IDF 핵심 키워드
# =========================================================

region_docs = (
    target_policies
    .groupby("지역_정규화")["분석텍스트_정제"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

tfidf_vectorizer = TfidfVectorizer(
    max_features=200,
    token_pattern=r"(?u)\b[가-힣A-Za-z0-9]{2,}\b"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(region_docs["분석텍스트_정제"])

try:
    tfidf_terms = tfidf_vectorizer.get_feature_names_out()
except:
    tfidf_terms = tfidf_vectorizer.get_feature_names()

rows = []

for i, region in enumerate(region_docs["지역_정규화"]):
    scores = tfidf_matrix[i].toarray().ravel()
    top_idx = scores.argsort()[::-1][:10]
    
    for rank, idx in enumerate(top_idx, start=1):
        if scores[idx] > 0:
            rows.append({
                "지역": region,
                "순위": rank,
                "키워드": tfidf_terms[idx],
                "TFIDF": scores[idx]
            })

region_tfidf = pd.DataFrame(rows)

region_tfidf.to_csv(
    TEXTMINING_DIR / "high_mismatch_region_tfidf_keywords.csv",
    index=False,
    encoding="utf-8-sig"
)

display(region_tfidf)

# 지역별 1위 키워드 그래프
top1 = (
    region_tfidf.sort_values(["지역", "TFIDF"], ascending=[True, False])
    .groupby("지역")
    .head(1)
    .copy()
)

top1["지역_키워드"] = top1["지역"] + " - " + top1["키워드"]
top1 = top1.sort_values("TFIDF", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top1["지역_키워드"], top1["TFIDF"])
plt.title("미스매치 지역별 TF-IDF 1위 키워드")
plt.xlabel("TF-IDF")
plt.tight_layout()
plt.savefig(TEXTMINING_DIR / "high_mismatch_region_tfidf_top1.png", dpi=200)
plt.close()

print("지역별 TF-IDF 분석 완료")

,지역,순위,키워드,TFIDF
0,전남,1,일자리,0.632881
1,전남,2,교육지원,0.363612
2,전남,3,직무교육,0.261407
3,전남,4,보조금,0.251580
4,전남,5,위한,0.145445
5,전남,6,중소기업,0.139548
6,전남,7,맞춤형상담서비스,0.131686
7,전남,8,복지,0.123824
8,전남,9,주거지원,0.113997
9,전남,10,통해,0.112032


지역별 TF-IDF 분석 완료


In [14]:
# =========================================================
# 6. TF-IDF 기반 K-means 정책 주제 유사 군집 분석
# =========================================================

policy_docs = target_policies.copy()
policy_docs = policy_docs[policy_docs["분석텍스트_정제"].str.len() > 0].copy()

n_docs = len(policy_docs)

if n_docs < 3:
    raise ValueError("K-means 군집화를 수행하기에는 정책 문서 수가 너무 적습니다.")

# 정책 수에 따라 군집 수 자동 조절
if n_docs >= 80:
    n_clusters = 4
elif n_docs >= 30:
    n_clusters = 3
else:
    n_clusters = 2

print("정책 문서 수:", n_docs)
print("K-means 군집 수:", n_clusters)

kmeans_tfidf = TfidfVectorizer(
    max_features=500,
    token_pattern=r"(?u)\b[가-힣A-Za-z0-9]{2,}\b",
    min_df=2
)

X = kmeans_tfidf.fit_transform(policy_docs["분석텍스트_정제"])

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

policy_docs["KMeans주제유사군집"] = kmeans.fit_predict(X)

try:
    feature_names = kmeans_tfidf.get_feature_names_out()
except:
    feature_names = kmeans_tfidf.get_feature_names()

# 군집별 대표 키워드
cluster_keyword_rows = []

centers = kmeans.cluster_centers_

for cluster_id in range(n_clusters):
    center = centers[cluster_id]
    top_idx = center.argsort()[::-1][:15]
    
    for rank, idx in enumerate(top_idx, start=1):
        cluster_keyword_rows.append({
            "KMeans주제유사군집": cluster_id,
            "순위": rank,
            "대표키워드": feature_names[idx],
            "중심값": center[idx]
        })

cluster_keywords = pd.DataFrame(cluster_keyword_rows)

# 군집별 정책 수
cluster_size = (
    policy_docs["KMeans주제유사군집"]
    .value_counts()
    .sort_index()
    .reset_index()
)
cluster_size.columns = ["KMeans주제유사군집", "정책수"]

# 군집별 대표 정책 예시
example_cols = ["KMeans주제유사군집", "지역_정규화"]

if policy_name_col is not None:
    example_cols.append(policy_name_col)

if policy_category_col is not None:
    example_cols.append(policy_category_col)

cluster_examples = (
    policy_docs[example_cols]
    .sort_values("KMeans주제유사군집")
    .groupby("KMeans주제유사군집")
    .head(5)
    .reset_index(drop=True)
)

# 저장
policy_docs.to_csv(
    TEXTMINING_DIR / "kmeans_topic_like_cluster_result.csv",
    index=False,
    encoding="utf-8-sig"
)

cluster_keywords.to_csv(
    TEXTMINING_DIR / "kmeans_topic_like_cluster_keywords.csv",
    index=False,
    encoding="utf-8-sig"
)

cluster_size.to_csv(
    TEXTMINING_DIR / "kmeans_topic_like_cluster_size.csv",
    index=False,
    encoding="utf-8-sig"
)

cluster_examples.to_csv(
    TEXTMINING_DIR / "kmeans_topic_like_cluster_examples.csv",
    index=False,
    encoding="utf-8-sig"
)

display(cluster_size)
display(cluster_keywords.head(30))
display(cluster_examples)

# 군집 크기 그래프
plt.figure(figsize=(8, 5))
plt.bar(cluster_size["KMeans주제유사군집"].astype(str), cluster_size["정책수"])
plt.title("미스매치 지역 정책의 K-means 주제 유사 군집 분포")
plt.xlabel("K-means 주제 유사 군집")
plt.ylabel("정책 수")
plt.tight_layout()
plt.savefig(TEXTMINING_DIR / "kmeans_topic_like_cluster_size.png", dpi=200)
plt.close()

print("K-means 기반 정책 주제 유사 군집 분석 완료")

정책 문서 수: 1069
K-means 군집 수: 4


,KMeans주제유사군집,정책수
0,0,82
1,1,536
2,2,138
3,3,313


,KMeans주제유사군집,순위,대표키워드,중심값
0,0,1,주거지원,0.418968
1,0,2,주거,0.067070
2,0,3,직무교육,0.049864
3,0,4,신혼부부,0.049056
4,0,5,공공임대주택,0.046744
5,0,6,대학생,0.042813
6,0,7,주거비,0.041750
7,0,8,지원사업,0.041030
8,0,9,대출이자,0.040434
9,0,10,대출,0.037988


,KMeans주제유사군집,지역_정규화,정책명,대표분류
0,0,전남,도심융합특구 조성,주거지원
1,0,전북,청년단기주거공간 운영(금강장),주거지원
2,0,전남,공공임대주택 공급,주거지원
3,0,전남,강진품애 청년 주거비 지원 사업,주거지원
4,0,전북,익산형 청년월세 지원사업,주거지원
5,1,전북,해외 K-Move 센터 운영,일자리
6,1,전북,전북특별자치도 기술창업 활성화 금융지원,일자리
7,1,전북,청년도전지원사업,일자리
8,1,전북,국내해외취업센터 운영,일자리
9,1,전북,국민내일배움카드,일자리


K-means 기반 정책 주제 유사 군집 분석 완료


In [15]:
# =========================================================
# 7. 군집 해석용 요약표 생성
# =========================================================

cluster_summary_rows = []

for cluster_id, part in cluster_keywords.groupby("KMeans주제유사군집"):
    top_words = part.sort_values("순위").head(8)["대표키워드"].tolist()
    top_words_text = ", ".join(top_words)
    
    cluster_count = cluster_size.loc[
        cluster_size["KMeans주제유사군집"] == cluster_id,
        "정책수"
    ].values[0]
    
    cluster_summary_rows.append({
        "KMeans주제유사군집": cluster_id,
        "정책수": cluster_count,
        "대표키워드": top_words_text,
        "해석용_군집명": ""
    })

cluster_summary = pd.DataFrame(cluster_summary_rows)

cluster_summary.to_csv(
    TEXTMINING_DIR / "kmeans_topic_like_cluster_summary_for_interpretation.csv",
    index=False,
    encoding="utf-8-sig"
)

display(cluster_summary)

,KMeans주제유사군집,정책수,대표키워드,해석용_군집명
0,0,82,"주거지원, 주거, 직무교육, 신혼부부, 공공임대주택, 대학생, 주거비, 지원사업",
1,1,536,"일자리, 중소기업, 보조금, 벤처, 창업, 맞춤형상담서비스, 위한, 인턴",
2,2,138,"복지, 출산, 보조금, 청년의, 바우처, 맞춤형상담서비스, 육아, 부담을",
3,3,313,"직무교육, 교육지원, 양성, 해외진출, 교육, 전문인력, 통한, 통해",


In [16]:
# =========================================================
# 8. 해석 파일 저장
# =========================================================

target_region_text = ", ".join(target_regions)

interpretation = f"""미스매치 지역 텍스트마이닝 분석 해석

1. 분석 목적

본 분석은 청년 유출-정책 미스매치가 높게 나타난 지역의 정책 텍스트를 별도로 추출하여, 해당 지역 정책들이 어떤 키워드와 주제 유사 군집으로 구성되어 있는지 확인하기 위해 수행하였다.

2. 분석 대상 지역

분석 대상 지역은 다음과 같다.

{target_region_text}

이 지역들은 청년유출위험도와 정책대응부족도를 기준으로 정책 미스매치 가능성이 높은 지역으로 선정하였다.

3. 분석 방법

본 분석에서는 LDA와 같은 확률적 토픽모델링은 사용하지 않았다. 대신 교재 범위에 맞추어 다음 방법을 사용하였다.

- 텍스트 전처리: 정책명, 정책설명, 정책키워드, 정책분류를 결합하고 불용어를 제거하였다.
- 키워드 빈도 분석: CountVectorizer를 활용하여 미스매치 지역 정책 텍스트의 주요 등장 단어를 확인하였다.
- TF-IDF 분석: 지역별 정책 텍스트에서 상대적으로 중요한 키워드를 추출하였다.
- K-means 군집 분석: TF-IDF 벡터를 기반으로 정책 텍스트가 유사한 정책끼리 묶이도록 군집화하였다.

4. K-means 분석의 의미

K-means 군집 분석은 정책 텍스트를 사람이 미리 정한 분류가 아니라 텍스트 유사도에 따라 자동으로 묶는 방법이다. 따라서 각 군집은 확률적 토픽모델링의 토픽이 아니라, '정책 주제 유사 군집'으로 해석해야 한다.

5. 해석상 주의점

K-means 군집 번호 자체에는 의미가 없다. 군집 0, 군집 1과 같은 번호는 분석 과정에서 임의로 부여된 값이므로, 각 군집의 대표 키워드와 대표 정책명을 확인한 뒤 사람이 군집명을 붙여야 한다.

6. 보고서 표현 예시

본 분석에서는 청년 유출-정책 미스매치가 높게 나타난 지역의 정책 텍스트를 대상으로 키워드 빈도 분석, TF-IDF 분석, K-means 기반 정책 주제 유사 군집 분석을 수행하였다. 이를 통해 미스매치 지역의 정책이 어떤 주제에 집중되어 있는지 확인하고, 청년 유출 문제와 정책 대응 간의 불균형을 해석하였다.
"""

(TEXTMINING_DIR / "interpretation.txt").write_text(interpretation, encoding="utf-8-sig")

print("해석 파일 저장 완료:", TEXTMINING_DIR / "interpretation.txt")

해석 파일 저장 완료: final_mismatch_results\03_mismatch_textmining\interpretation.txt


In [17]:
print("03_mismatch_textmining 결과 파일 목록")

for f in sorted(TEXTMINING_DIR.iterdir()):
    print("-", f.name)

03_mismatch_textmining 결과 파일 목록
- high_mismatch_keyword_frequency.csv
- high_mismatch_keyword_frequency.png
- high_mismatch_policy_dataset.csv
- high_mismatch_region_tfidf_keywords.csv
- high_mismatch_region_tfidf_top1.png
- interpretation.txt
- kmeans_topic_like_cluster_examples.csv
- kmeans_topic_like_cluster_keywords.csv
- kmeans_topic_like_cluster_result.csv
- kmeans_topic_like_cluster_size.csv
- kmeans_topic_like_cluster_size.png
- kmeans_topic_like_cluster_summary_for_interpretation.csv
- target_mismatch_regions.csv
